In [4]:
"""
test_qwen3_raw.py
=================
Minimal script to inspect Qwen3-8B response structure.
Uses Ollama — no vLLM server needed.

Setup:
    1. Install Ollama: https://ollama.com
    2. Pull model:  ollama pull qwen3:8b
    3. Run:         python test_qwen3_raw.py
"""

import json
import re
import requests
from dataclasses import dataclass
from typing import Optional

BASE_URL = "http://localhost:11434"   # Ollama default port

# =============================================================================
# STEP 1 — Confirm model is available
# =============================================================================

# print("=" * 60)
# print("STEP 1: Check available models")
# print("=" * 60)

r = requests.get(f"{BASE_URL}/api/tags")
models = r.json()
# print(json.dumps(models, indent=2))

# =============================================================================
# STEP 2 — Raw call, no schema, thinking ON
# =============================================================================

# print("\n" + "=" * 60)
# print("STEP 2: Raw call — thinking ON, no schema")
# print("=" * 60)

# payload = {
#     "model":  "qwen3:8b",
#     "prompt": "What is 2 + 2? Answer briefly.",
#     "stream": False,
#     "options": {"temperature": 0.0},
# }

# r    = requests.post(f"{BASE_URL}/api/generate", json=payload, timeout=120)
# data = r.json()

# print("\n--- Full response structure ---")
# print(json.dumps(data, indent=2))

# raw = data["response"]
# print("\n--- Raw content (repr) ---")
# print(repr(raw))

# =============================================================================
# STEP 3 — Chat call with system prompt, thinking ON
# =============================================================================

print("\n" + "=" * 60)
print("STEP 3: Chat call — system prompt + thinking ON")
print("=" * 60)

payload_chat = {
    "model":  "qwen3:8b",
    "messages": [
        {"role": "system",  "content": "You are a helpful assistant. /think"},
        {"role": "user",    "content": "Name one nonviolent protest tactic."},
    ],
    "stream": False,
    "options": {"temperature": 0.0},
}

r         = requests.post(f"{BASE_URL}/api/chat", json=payload_chat, timeout=120)
data_chat = r.json()

print("\n--- Full chat response structure ---")
print(json.dumps(data_chat, indent=2))

raw_chat = data_chat["message"]["content"]
print("\n--- Raw content (repr) ---")
print(repr(raw_chat))
print("\n--- Raw content (printed) ---")
print(raw_chat)

# =============================================================================
# STEP 4 — Chat call asking for JSON output
# =============================================================================

import json
import re
import requests
from dataclasses import dataclass
from typing import Optional

BASE_URL = "http://localhost:11434"   # Ollama default port
 
print("\n" + "=" * 60)
print("STEP 1: Chat call — JSON output requested")
print("=" * 60)

payload_json = {
    "model":  "qwen3:8b",
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a nonviolent activist agent. "
                "Respond ONLY with valid JSON, nothing else.\n"
                "Use this exact structure:\n"
                "{\n"
                '  "decision": {\n'
                '    "chosen_position": [x, y],\n'
                '    "target_agent_group": "citizens | pillars | police | activists",\n'
                '    "target_pillar_institution": "N/A or institution name",\n'
                '    "primary_attribute": "Nonviolence",\n'
                '    "call_to_action_type": "sit_in | march | boycott | strike | civil_disobedience | petition"\n'
                "  }\n"
                "}"
            ),
        },
        {
            "role": "user",
            "content": "Available positions: [[3,4],[4,5],[2,3]]. Target citizens. Choose a position and tactic.",
        },
    ],
    # "format": "json",    # Ollama native JSON mode
    "stream": False,
    "options": {"temperature": 0.0, "num_tokens": 512},
}

r          = requests.post(f"{BASE_URL}/api/chat", json=payload_json, timeout=120)
data_json  = r.json()


# raw_json = data_json["message"]["content"]
# print("\n--- Raw content (repr) ---")
# print(repr(raw_json))
# print("\n--- Raw content (printed) ---")
# print(raw_json)

# =============================================================================
# STEP 5 — Run extract_thinking on the JSON response
# =============================================================================

print("\n" + "=" * 60)
print("STEP 5: What _extract_thinking produces")
print("=" * 60)

@dataclass
class LLMResponse:
    thinking: Optional[str]
    content:  str

def extract_thinking(raw: str) -> LLMResponse:
    think_match = re.search(r"<think>(.*?)</think>", raw, flags=re.DOTALL)
    thinking    = think_match.group(1).strip() if think_match else None
    content     = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    content     = re.sub(r"^```(?:json)?\s*", "", content)
    content     = re.sub(r"\s*```$",           "", content)
    return LLMResponse(thinking=thinking, content=content.strip())

parsed = extract_thinking(raw_json)

# print(f"\nthinking present : {parsed.thinking is not None}")
# print(f"thinking length  : {len(parsed.thinking) if parsed.thinking else 0} chars")
# print(f"\n--- thinking (first 300 chars) ---")
# print((parsed.thinking or "None")[:300])
# print(f"\n--- content (should be clean JSON) ---")
# print(parsed.content)

# try:
#     obj = json.loads(parsed.content)
#     print("\n  content is valid JSON")
#     print(json.dumps(obj, indent=2))
# except json.JSONDecodeError as e:
#     print(f"\n  content is NOT valid JSON: {e}")


STEP 3: Chat call — system prompt + thinking ON

--- Full chat response structure ---
{
  "error": "model 'qwen3:8b' not found"
}


KeyError: 'message'

In [ ]:
python -m vllm.entrypoints.openai.api_server \
    --model <model_path_or_name> \
    --port 8000 \
    --host 0.0.0.0

In [ ]:
import subprocess
import os

# 1. THE WSL PATH (Crucial)
model_path = "/mnt/c/Users/fmahnoor1/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218"

# 2. Local GPU Strategy
# A local GPU (like an RTX 3080/4090) has 10-24GB VRAM. 
# We need to be a bit more conservative than the cluster.
command = [
    "wsl", "python3", "-m", "vllm.entrypoints.openai.api_server", # Added "wsl"
    "--model", model_path,
    "--dtype", "half",
    "--max-model-len", "2048",
    "--max-num-seqs", "1",
    "--gpu-memory-utilization", "0.9",
    "--port", "8000",
    "--host", "0.0.0.0", 
    "--trust-remote-code"
]

# Launch
log_file = open("vllm_local.log", "w")
process = subprocess.Popen(command, stdout=log_file, stderr=log_file)
print(f"🚀 Local vLLM started! PID: {process.pid}")

🚀 Local vLLM started! PID: 62032


In [ ]:
/home/fmahnoor1/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c

In [ ]:
wsl.exe /home/fmahnoor1/vllm-env/bin/python3 -m vllm.entrypoints.openai.api_server --model "/mnt/c/Users/fmahnoor1/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c" --dtype half --max-model-len 512 --gpu-memory-utilization 0.95 --enforce-eager --max-num-seqs 1 --port 8000 --host 0.0.0.0 --trust-remote-code

In [ ]:
wsl.exe /home/fmahnoor1/vllm-env/bin/python3 -m vllm.entrypoints.openai.api_server --model "/mnt/c/Users/fmahnoor1/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c" --dtype auto --max-model-len 512 --gpu-memory-utilization 0.8 --port 8000 --host 0.0.0.0 --trust-remote-code

wsl sudo chmod -R 755 /mnt/c/Users/fmahnoor1/.cache/huggingface

Test-NetConnection -ComputerName localhost -Port 8000

In [ ]:
import time
import requests

# --- 1. WAIT FOR SERVER ---
print("⏳ Checking if vLLM server is ready...")
ready = False
for _ in range(3):  # Wait up to 150 seconds
    try:
        # vLLM is ready when the /v1/models endpoint returns 200
        response = requests.get("http://localhost:8000/v1/models", timeout=300)
        if response.status_code == 200:
            print("✅ vLLM Server is LIVE!")
            ready = True
            break
    except requests.exceptions.ConnectionError:
        pass
    print(".", end="", flush=True)
    time.sleep(5)


if not ready:
    raise RuntimeError("vLLM server failed to start. Check vllm_local.log for OOM or path errors.")


⏳ Checking if vLLM server is ready...
...

RuntimeError: vLLM server failed to start. Check vllm_local.log for OOM or path errors.

In [1]:

import os
from run_simulation import RunOrchestrator, BASE_PARAMS
from Logging.log_config import LogConfig

# --- 2. RUN SIMULATION ---
# os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

LOG_CFG = LogConfig(
    provider    = "vllm",
    prompt_type = "ZeroShot",
    memory      = "stateful",
    vision      = "vision-4",
    base_dir    = "SimulationLogs/vllm_currentzeroshot",
)

model_params = dict(BASE_PARAMS)

orchestrator = RunOrchestrator(
    base_config  = LOG_CFG,
    params       = model_params,
    max_workers  = 1, 
    start_run_id = 1, 
)

summary = orchestrator.run(n_runs=1)

[RunOrchestrator] Starting 1 runs (sequential)
  Run    1 / 1  (seed=1)  

c:\Users\fmahnoor1\AppData\Local\anaconda3\envs\abm-llm\Lib\site-packages\solara\validate_hooks.py:122: UserWarning: c:\Users\fmahnoor1\AppData\Local\anaconda3\envs\abm-llm\Lib\site-packages\mesa\experimental\jupyter_viz.py:164: JupyterViz: `use_state` found within a nested function created on line 156
To suppress this check, replace the line with:
        grid_layout, set_grid_layout = solara.use_state(grid_layout_initial)  # noqa: SH104

Make sure you understand the consequences of this, by reading about the rules of hooks at:
    https://solara.dev/documentation/advanced/understanding/rules-of-hooks

  warnings.warn(str(e))


[VLLMProvider] Scanning ports 8000–8020 for vLLM...
[VLLMProvider] Found server at http://localhost:8000
[VLLMProvider] Resolved model: /mnt/c/Users/fmahnoor1/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c
[VLLMProvider] Ready — url=http://localhost:8000  model=/mnt/c/Users/fmahnoor1/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c  prompt_type=ZeroShot

🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a position that strengthens nonviolent resistance. The current neighborhood data shows various directions with different citizen conditions, police presence, and activist activities.

First, the NEIGHBORHOOD ASSESSMENT. The most critical directions are probably where there are suffering citizens and minimal police. Looking at the data, the southeast has 2 suffering citizens and no police. The southwest also has 2 suffering


🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a position that strengthens non-violent resistance. The previous move was towards pillars to focus on empathy. Now, the neighborhood state shows that the northeast has a stable pillar, the Business Community. The southwest has some suffering citizens. The goal is to use Gandhi's strategies, so empathy and truth are key.

Looking at the directions, the northeast has a pillar, which aligns with the previous focus on truth. The southwest has citizens in suffering, which relates to empathy. But the user's last move was towards pillars, so maybe the next step is to engage with the Business Community. However, the current position is (25,36), and the can_move_to in northeast is (23,38). But the southwest has positions like (27,34). Wait, the user needs to choose the best position based on the current state. The northeast has a pillar, which is a stable institution. The sout


🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a specific position to strengthen nonviolent resistance. The key is to follow Gandhi's principles.

First, looking at the neighborhood assessment. The most critical directions are southeast and northwest. Southeast has 3 suffering citizens, and northwest has 2 suffering citizens plus a police officer. But the northwest also has a stable pillar, which might be important for building alliances.

Gandhi's core attributes here would be empathy, as the focus is on the suffering citizens. The risk is the police presence in northwest, but the goal is to reach the citizens. The decision is to move towards the southeast, where there are more suffering citizens. The target group is citizens, and the primary attribute is empathy. The call to action would be a sit-in to show solidarity.
--------------------------------------------------
📦 FINAL CONTENT (JSON):
{
  "decision": {
 


🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a specific position to strengthen non-violent resistance. The current neighborhood state shows various directions with different agents.

First, the neighborhood assessment. The most critical directions are southwest and southeast. Southwest has 5 suffering citizens, which is high. Southeast has 3 suffering citizens. Both are important for empathy. 

Gandhian prioritization here is empathy because the goal is to reach out to the suffering. The previous move was towards citizens to practice empathy. Now, the southwest direction has the most suffering citizens. 

Risk/benefit: Moving to southwest has a high number of suffering citizens but also has a stable pillar, which might be a resource. However, there's no police presence there, so it's safer. 

Decision: Move to southwest, target citizens. The primary attribute is empathy. The call to action is to reach out to the


🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a specific position to strengthen non-violent resistance. The key is to follow Gandhi's principles, so I need to focus on empathy, truth, and nonviolence.

Looking at the neighborhood data, the adjacent area has a police officer active. That's a risk, but maybe there's a way to engage with the citizens who are suffering. The northwest direction has 2 suffering citizens. Moving there could help address their needs, aligning with empathy. Also, the pillars in the adjacent area are stable, but since the target group is citizens, maybe that's not the focus. The primary attribute here is empathy. The call to action would be to reach out to the suffering citizens. So the chosen position is (28,4) or (29,4) in northwest. Let's pick (28,4) as the position. Target group is citizens, primary attribute is empathy, and the call to action is sit_in.
-------------------------------


🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a specific position to strengthen nonviolent resistance. The current neighborhood state shows that the adjacent area has 4 unbothered citizens and 1 suffering citizen, along with active police. The northwest has 2 suffering citizens and 1 active police. The southeast has 1 suffering citizen. The southwest has 6 unbothered. The west has 1 suffering. The south has 1 suffering. The east and north have no police but some citizens.

The previous move was towards citizens to focus on empathy. Now, the most critical directions are northwest and adjacent, as they have the highest number of suffering citizens and active police. The core attribute here is empathy, as we need to address the suffering. The risk is the active police, but the benefit is engaging the suffering citizens. The decision is to move to the adjacent area's can_move_to positions, targeting citizens to apply


🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a specific position to strengthen nonviolent resistance. The key is to follow Gandhi's principles.

First, looking at the neighborhood assessment. The most critical directions are southwest and northwest. Southwest has 4 suffering citizens, which is a big concern. Northwest has 2 suffering citizens as well. These areas need attention because of the empathy aspect.

Gandhi's core attribute here would be Empathy, as the focus is on the suffering citizens. The risk-benefit analysis: moving towards southwest might face police, but there's a sympathetic police officer. However, the main goal is to reach the citizens. 

The decision is to move to southwest, targeting citizens. The primary attribute is Empathy. The call to action is a sit-in to show solidarity. The target group is citizens, and since it's not pillars, target_pillar_institution is N/A.
-----------------------


🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a specific position to strengthen nonviolent resistance. The previous move was towards citizens to show empathy. Now, looking at the current neighborhood state, the key directions are southeast and adjacent. 

In the southeast, there are citizens protesting and some jailed. The adjacent area has sympathetic police and suffering citizens. The main goal is to use empathy and truth to help those in need. The southeast has active protesters, so moving there could engage more people. But the adjacent area has sympathetic police, which might be a risk. However, the primary attribute here is empathy, so focusing on the suffering citizens in the adjacent area makes sense. The chosen position would be (9,3) from adjacent. Targeting citizens to show empathy and support. The call to action is a sit-in to demonstrate commitment.
--------------------------------------------------



🤖 AGENT REASONING (THINKING):
Okay, let's break this down. The user is in a simulation where they need to move towards a specific position to strengthen nonviolent resistance. The previous move was towards citizens to show empathy. Now, looking at the current neighborhood state, the key directions are southeast and southwest. 

In the southeast, there are 2 suffering citizens and 1 jailed person. The southwest has 2 suffering citizens and 2 unbothered, but there's a sympathetic police officer. The adjacent area has 4 unbothered citizens and a sympathetic police. The northeast has a protesting citizen, which might be a good target for a march.

Gandhi's principles emphasize empathy and courage. The southwest has suffering citizens, so moving there to engage with them would align with empathy. Also, the sympathetic police might be a resource. The primary attribute here is empathy. The call to action could be a march towards the protesting citizens in the northeast, but since the southwe

In [ ]:
7900/3600